In [ ]:
!pip install -q gensim tensorflow scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
num_words = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=num_words
)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))


In [ ]:
word_index = imdb.get_word_index()

# Reverse dictionary
reverse_word_index = {
    value: key for key, value in word_index.items()
}

def decode_review(review):
    words = []

    for i in review:
        if i == 0:
            words.append("<PAD>")
        elif i == 1:
            words.append("<START>")
        elif i == 2:
            words.append("<UNK>")
        elif i >= 3:
            words.append(reverse_word_index.get(i - 3, "<UNK>"))

    return words


In [ ]:
train_sentences = [
    decode_review(review)
    for review in x_train
]

print(train_sentences[0][:20])


In [ ]:
word2vec_model = Word2Vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

print("Word2Vec vocabulary size:",
      len(word2vec_model.wv))


Word2Vec vocabulary size: 9998


In [ ]:
embedding_dim = 100

embedding_matrix = np.zeros(
    (num_words, embedding_dim)
)

for word, index in word_index.items():

    if index < num_words:

        if word in word2vec_model.wv:

            embedding_matrix[index] = \
                word2vec_model.wv[word]


print("Embedding matrix shape:",
      embedding_matrix.shape)


Embedding matrix shape: (10000, 100)


In [ ]:

max_length = 200

x_train_pad = pad_sequences(
    x_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print("Training shape:", x_train_pad.shape)
print("Testing shape:", x_test_pad.shape)


Training shape: (25000, 200)
Testing shape: (25000, 200)


In [ ]:
model = Sequential([

    Embedding(
        input_dim=num_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=False
    ),

    LSTM(128),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dropout(0.5),

    Dense(1, activation="sigmoid")
])


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)

In [ ]:
history = model.fit(
    x_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
loss, accuracy = model.evaluate(
    x_test_pad,
    y_test
)

print("\nTest Accuracy:", accuracy)


In [ ]:
y_pred_probability = model.predict(
    x_test_pad
)

y_pred = (y_pred_probability > 0.5).astype(int)


In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

print("\n========== PERFORMANCE ==========")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


In [ ]:
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"]
    )
)


In [ ]:

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")

plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.show()

Question 2: Data Augmentation and Analysis

In [ ]:
# ============================================
# NLP DATA AUGMENTATION + PERFORMANCE ANALYSIS
# ============================================

import numpy as np
import random
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [ ]:
# --------------------------------------------
# 1. Load Dataset
# --------------------------------------------

num_words = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=num_words
)

print("Original training data:", len(x_train))
print("Testing data:", len(x_test))


In [ ]:
# --------------------------------------------
# 2. Decode IMDB Reviews
# --------------------------------------------

word_index = imdb.get_word_index()

reverse_word_index = {
    value: key for key, value in word_index.items()
}


def decode_review(review):

    words = []

    for i in review:

        if i == 0:
            continue

        elif i == 1:
            words.append("<START>")

        elif i == 2:
            words.append("<UNK>")

        else:
            words.append(
                reverse_word_index.get(
                    i - 3,
                    "<UNK>"
                )
            )

    return " ".join(words)


train_text = [
    decode_review(review)
    for review in x_train
]

test_text = [
    decode_review(review)
    for review in x_test
]


In [ ]:
# --------------------------------------------
# 3. Data Augmentation
# --------------------------------------------

def random_deletion(sentence, probability=0.1):

    words = sentence.split()

    # Don't modify very short sentences
    if len(words) <= 3:
        return sentence

    new_words = []

    for word in words:

        # Randomly delete word
        if random.random() > probability:
            new_words.append(word)

    # Make sure sentence doesn't become empty
    if len(new_words) == 0:
        return random.choice(words)

    return " ".join(new_words)

In [ ]:
# --------------------------------------------
# 4. Generate Augmented Dataset
# --------------------------------------------

augmented_text = []

for text in train_text:

    augmented_text.append(
        random_deletion(
            text,
            probability=0.1
        )
    )


print("\nExample Original:")
print(train_text[0])

print("\nExample Augmented:")
print(augmented_text[0])

In [ ]:
# --------------------------------------------
# 5. Tokenization
# --------------------------------------------

tokenizer = Tokenizer(
    num_words=num_words
)

# Fit on original + augmented text
tokenizer.fit_on_texts(
    train_text + augmented_text
)


In [ ]:
# --------------------------------------------
# 6. Convert Text to Sequences
# --------------------------------------------

original_sequences = tokenizer.texts_to_sequences(
    train_text
)

augmented_sequences = tokenizer.texts_to_sequences(
    augmented_text
)

test_sequences = tokenizer.texts_to_sequences(
    test_text
)

In [ ]:

# --------------------------------------------
# 7. Padding
# --------------------------------------------

max_length = 200

x_original = pad_sequences(
    original_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

x_augmented = pad_sequences(
    augmented_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

x_test = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [ ]:
# --------------------------------------------
# 8. Function to Create LSTM Model
# --------------------------------------------

vocab_size = num_words

def create_model():

    model = Sequential([

        Embedding(
            input_dim=vocab_size,
            output_dim=100,
            input_length=max_length
        ),

        LSTM(128),

        Dropout(0.5),

        Dense(64, activation="relu"),

        Dropout(0.5),

        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [ ]:
# ============================================
# MODEL 1: ORIGINAL DATA
# ============================================

print("\n==============================")
print("TRAINING ON ORIGINAL DATA")
print("==============================")


model_original = create_model()

history_original = model_original.fit(
    x_original,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)


In [ ]:
# --------------------------------------------
# Evaluate Original Model
# --------------------------------------------

prob_original = model_original.predict(
    x_test
)

pred_original = (
    prob_original > 0.5
).astype(int)


original_accuracy = accuracy_score(
    y_test,
    pred_original
)

original_precision = precision_score(
    y_test,
    pred_original
)

original_recall = recall_score(
    y_test,
    pred_original
)

original_f1 = f1_score(
    y_test,
    pred_original
)



In [ ]:

# ============================================
# MODEL 2: AUGMENTED DATA
# ============================================

print("\n==============================")
print("TRAINING ON AUGMENTED DATA")
print("==============================")


model_augmented = create_model()

history_augmented = model_augmented.fit(
    x_augmented,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
prob_augmented = model_augmented.predict(
    x_test
)

pred_augmented = (
    prob_augmented > 0.5
).astype(int)


augmented_accuracy = accuracy_score(
    y_test,
    pred_augmented
)

augmented_precision = precision_score(
    y_test,
    pred_augmented
)

augmented_recall = recall_score(
    y_test,
    pred_augmented
)

augmented_f1 = f1_score(
    y_test,
    pred_augmented
)

In [ ]:

# ============================================
# PERFORMANCE COMPARISON
# ============================================

print("\n========================================")
print("       PERFORMANCE COMPARISON")
print("========================================")

print(
    f"{'Metric':<15}"
    f"{'Original':<15}"
    f"{'Augmented':<15}"
)

print("-" * 45)

print(
    f"{'Accuracy':<15}"
    f"{original_accuracy:<15.4f}"
    f"{augmented_accuracy:<15.4f}"
)

print(
    f"{'Precision':<15}"
    f"{original_precision:<15.4f}"
    f"{augmented_precision:<15.4f}"
)

print(
    f"{'Recall':<15}"
    f"{original_recall:<15.4f}"
    f"{augmented_recall:<15.4f}"
)

print(
    f"{'F1 Score':<15}"
    f"{original_f1:<15.4f}"
    f"{augmented_f1:<15.4f}"
)


In [ ]:

# ============================================
# PERFORMANCE GRAPH
# ============================================

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

original_scores = [
    original_accuracy,
    original_precision,
    original_recall,
    original_f1
]

augmented_scores = [
    augmented_accuracy,
    augmented_precision,
    augmented_recall,
    augmented_f1
]


x = np.arange(len(metrics))

width = 0.35

plt.figure(figsize=(9, 6))

plt.bar(
    x - width/2,
    original_scores,
    width,
    label="Original Data"
)

plt.bar(
    x + width/2,
    augmented_scores,
    width,
    label="Augmented Data"
)

plt.xticks(
    x,
    metrics
)

plt.ylabel("Score")

plt.title(
    "Original vs Augmented Dataset Performance"
)

plt.ylim(0, 1)

plt.legend()

plt.show()


In [ ]:
# ============================================
# TRAINING ACCURACY COMPARISON
# ============================================

plt.figure(figsize=(9, 5))

plt.plot(
    history_original.history["val_accuracy"],
    label="Original Validation Accuracy"
)

plt.plot(
    history_augmented.history["val_accuracy"],
    label="Augmented Validation Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Validation Accuracy")

plt.title(
    "Validation Accuracy Comparison"
)

plt.legend()

plt.show()

In [ ]:
# ============================================
# TRAINING LOSS COMPARISON
# ============================================

plt.figure(figsize=(9, 5))

plt.plot(
    history_original.history["val_loss"],
    label="Original Validation Loss"
)

plt.plot(
    history_augmented.history["val_loss"],
    label="Augmented Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Validation Loss")

plt.title(
    "Validation Loss Comparison"
)

plt.legend()

plt.show()